Start time of execution

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymongo
import yaml
from time import time

from utils import sliding_window_pd, filter_instances
from utils_visual import (
    plot_filtered_vs_raw_signal,
    plot_feature_boxplots_by_class,
    plot_scatter_pca,
    plot_window_counts_by_gesture_user
)
from utils_features import extract_all_candidates

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

**Load Configuation**

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
time_start = time()


In [ ]:
client = pymongo.MongoClient(config["client"])
db = client[config["db"]]
collection = db[config["col"]]


In [ ]:
found_keys = collection.distinct("_id")
print("Total number of documents in the collection:", len(found_keys))

In [ ]:
documents = collection.find({"_id": {"$in": found_keys}})
rows = []
for doc in documents:
    n_samples = len(doc["data"]["acc_x"])
    for i in range(n_samples):
        rows.append({
            "acc_x": float(doc["data"]["acc_x"][i]),
            "acc_y": float(doc["data"]["acc_y"][i]),
            "acc_z": float(doc["data"]["acc_z"][i]),
            "gyr_x": float(doc["data"]["gyr_x"][i]),
            "gyr_y": float(doc["data"]["gyr_y"][i]),
            "gyr_z": float(doc["data"]["gyr_z"][i]),
            "gesture_id": doc["gesture_id"],

            "user": doc["user"],
        })

dataframe = pd.DataFrame(rows)
# Should be float/int
print(dataframe[["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "gesture_id","user"]].dtypes)
print(dataframe[:5])

## Data Exploration
This section will explore the dataset to understand its structure, identify any missing values, and analyze the distribution of the target variable.

### Step 1: Barplot of each recording

In [ ]:
dict = {}
for key,value in dataframe.groupby(["gesture_id", "user"]):
    new_key = f"{key[0]}_{key[1]}"
    dict[new_key] = len(value) / 100 # sampling rate is 100Hz, so we divide by 100 to get the count in seconds
plt.figure(figsize=(12, 6))
sns.barplot(x=list(dict.keys()), y=list(dict.values()))
plt.xticks(rotation=90)
plt.xlabel("Gesture ID and User")
plt.ylabel("Count (in seconds)")
plt.title("Count of Each Gesture ID and User")
plt.show()


### Step 2: Dataset overview

### Step 4: Filter Signals

In [ ]:
list_of_signals = [value for key,value in dataframe.groupby(["gesture_id","user"])]
filtered_list_of_signals = filter_instances(
        instances_list = list_of_signals,
        order=config["filter"]["order"],
        filter_type=config["filter"]["type"],
        wn = config["filter"]["wn"]
    )

In [ ]:
plot_filtered_vs_raw_signal(
    raw_signals = list_of_signals,
    filtered_signals = filtered_list_of_signals,
    idx = 0,
    n_start = 0,
    n_end = 100
)

### Step 5: Windowing

In [ ]:
# Windowing
windows = []
for signal in filtered_list_of_signals:
    temp_windows = sliding_window_pd(
        df=signal,
        ws=config["sliding_window"]["ws"],
        overlap=config["sliding_window"]["overlap"],
        w_type=config["sliding_window"]["w_type"],
    )
    for window in temp_windows:
        windows.append(window)

print(f"Total number of windows: {len(windows)}")
print(f"Shape of first window: {windows[0].shape}")

### Step 6: Windows Boxplot


In [ ]:
plot_window_counts_by_gesture_user(windows)


# Data Transformation and Feature Engineering

In [ ]:
features_df = pd.DataFrame([extract_all_candidates(window) for window in windows])
print("Extracted features shape:", features_df.shape)
print("First few rows of the features dataframe:")
print(features_df.head())

## Different Train/Test Split
To further evaluate the robustness of our model, we will perform a different train/test split. This time, we will use a stratified split and each set will contain data from 3 users. This will allow us to see if the previously selected model and features generalize well to a different subset of users. We will repeat the same evaluation process as before to compare the results with the previous split.

In [ ]:
all_features = pd.concat([features_df], axis=0).reset_index(drop=True)

X_all = all_features.drop(columns=["user", "gesture_id"])
y_all = all_features["gesture_id"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all,
    y_all,
    test_size=0.25,
    stratify=y_all,
    random_state=42
)
y_tr  

In [ ]:
selector = SelectKBest(f_classif, k=10)
selector.fit(X_tr, y_tr)
scores   = pd.Series(selector.scores_, index=X_tr.columns).sort_values(ascending=False)
selected = X_tr.columns[selector.get_support()].tolist()
print("score: ")
print(scores.head(10))
print("ANOVA F-scores (top 10):")
print(selected)

In [ ]:
x_tr_selected = X_tr[selected].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(x_tr_selected, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Selected Features (Train Set)", fontsize=16, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plot_feature_boxplots_by_class(
    df=pd.concat([X_tr[selected], y_tr], axis=1),
    features=selected,
    label_col="gesture_id",
    figsize_per_plot=(6, 5)
)

In [ ]:
cols_to_drop = ["acc_y_mean", "acc_mag_mean"]
X_tr_reduced = X_tr[selected].drop(columns=cols_to_drop)
X_te_reduced = X_te[selected].drop(columns=cols_to_drop)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_reduced)
X_te_scaled = scaler.transform(X_te_reduced)

In [ ]:
X_tr_pca = PCA(n_components=2, random_state=42).fit_transform(X_tr_scaled)
pca_df = pd.DataFrame(X_tr_pca, columns=["PC1", "PC2"])
binary_labels = y_tr.astype("category").cat.codes
pca_df["binary_label"] = binary_labels.values
from utils_visual import plot_scatter_pca
plot_scatter_pca(
    df=pca_df,
    c_name="binary_label"
)

In [ ]:
svc = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svc.fit(X_tr_scaled, y_tr)
y_pred_svc = svc.predict(X_te_scaled)
random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
random_forest.fit(X_tr_reduced, y_tr)
y_pred_rf = random_forest.predict(X_te_reduced)
from sklearn.metrics import classification_report
print("SVC Classification Report:")
print(classification_report(y_te, y_pred_svc))
print("Random Forest Classification Report:")
print(classification_report(y_te, y_pred_rf))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig,ax = plt.subplots(figsize=(8, 6))
fig.suptitle("Confusion Matrix for SVC", fontsize=16, fontweight="bold")
confusion_svc = confusion_matrix(y_te, y_pred_svc,labels=svc.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_svc, display_labels=svc.classes_)
disp.plot(ax=ax, xticks_rotation=45,  colorbar=False)
fig,ax = plt.subplots(figsize=(8, 6))
fig.suptitle("Confusion Matrix for Random Forest", fontsize=16, fontweight="bold")
confusion_rf = confusion_matrix(y_te, y_pred_rf,labels=random_forest.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_rf, display_labels=random_forest.classes_)
disp.plot(ax=ax, xticks_rotation=45,  colorbar=False)

### Experiment evaluation
The 1st split (subject independent) performed poorly, with an accuracy of around 0.18, which is even worse than random guessing (0.2 for 5 classes)!
That proved that the features we extracted were not generalizing well to unseen users, which is a common issue in time series classification problems, especially when dealing with human activity recognition data. The model was likely overfitting to the specific patterns of the users in the training set and could not capture the underlying patterns that are common across different users.
One other factor is that the subjects in the training set were not representative of the subjects in the test set, which can also lead to poor generalization. This highlights the importance of having a diverse and representative training set when working with time series data, especially in the context of human activity recognition.


On the other hand, as expected, the 2nd split (contained all the subjects participating in both sets) performed much better, with an accuracy of around 0.8. This is because the model was able to learn from the patterns present in the training set that were also present in the test set, leading to better generalization. However, this also indicates that the model may not be robust to unseen users, which is a common challenge in time series classification problems. It is important to consider this when deploying models in real-world applications, as they may encounter new users with different patterns that were not seen during training.

### Cross Validation on 2nd split
To further validate the performance of our model on the 2nd split, we will perform cross-validation. This will involve splitting the training data into multiple folds and training the model on different combinations of these folds while evaluating on the remaining fold. This process will be repeated multiple times to ensure that our results are consistent and not due to random chance. We will report the average performance metrics across all folds to provide a more robust estimate of our model's performance on unseen data.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
param_grid = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [None, 10, 20],
    "rf__min_samples_leaf": [1, 3, 5]
}

pipeline_steps = [
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1)
    )
]
grid = GridSearchCV(Pipeline(pipeline_steps), param_grid,cv=5,scoring='accuracy', n_jobs=-1)
grid.fit(X_tr_reduced, y_tr)
y_pred_grid = grid.predict(X_te_reduced)
print("Best parameters found: ", grid.best_params_)
print("Classification Report for Random Forest (Grid Search):")
print(classification_report(y_te, y_pred_grid))
fig,ax = plt.subplots(figsize=(8, 6))
confusion_grid = confusion_matrix(y_te, y_pred_grid,labels=grid.best_estimator_.named_steps['rf'].classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_grid, display_labels=grid.best_estimator_.named_steps['rf'].classes_)
disp.plot(ax=ax, xticks_rotation=45,  colorbar=False)

In [ ]:
time_end = time()
elapsed_time = time_end - time_start
print(f"Total execution time: {elapsed_time:.2f} seconds")